# Programming Assignment: Statistical Feature Selection

Welcome to the Statistical Feature Selection Programming Assignment!

Feature selection is the process of identifying and keeping only the most relevant features for your model. This reduces noise, speeds up training, and often improves generalisation.

**What You Will Do in this Assignment:**

* **Filter Methods** – Apply variance threshold, correlation-based, and mutual information selection
* **Statistical Tests Selection** – Use SelectKBest with f_classif, f_regression, and chi2 for different problem types
* **Correlation-Based Elimination** – Systematically remove highly correlated (redundant) features
* **Feature Selection Pipeline** – Build a complete workflow: scale → select → model
* **From-Scratch Implementation** – Implement the ANOVA F-statistic manually

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents
- [Imports](#imports)
- [1 - Filter Methods](#1)
    - [1.1 - Load Data](#1-1)
    - [1.2 - Variance Threshold](#1-2)
        - **[Exercise 1 - apply_variance_threshold](#ex-1)**
    - [1.3 - Visualise Feature Variances](#1-3)
        - **[Exercise 2 - plot_variance_scores](#ex-2)**
    - [1.4 - Mutual Information Selection](#1-4)
        - **[Exercise 3 - select_with_mutual_information](#ex-3)**
- [2 - Statistical Tests Selection](#2)
    - [2.1 - F-test for Classification](#2-1)
        - **[Exercise 4 - select_with_f_classif](#ex-4)**
    - [2.2 - Chi-Square Test](#2-2)
        - **[Exercise 5 - select_with_chi2](#ex-5)**
    - [2.3 - F-test for Regression](#2-3)
        - **[Exercise 6 - select_for_regression](#ex-6)**
    - [2.4 - Compare Selection Methods](#2-4)
        - **[Exercise 7 - compare_selection_methods](#ex-7)**
- [3 - Correlation-Based Elimination](#3)
    - [3.1 - Find Correlated Pairs](#3-1)
        - **[Exercise 8 - find_correlated_pairs](#ex-8)**
    - [3.2 - Remove Correlated Features](#3-2)
        - **[Exercise 9 - remove_correlated_features](#ex-9)**
- [4 - Feature Selection Pipeline](#4)
    - [4.1 - Build Complete Pipeline](#4-1)
        - **[Exercise 10 - create_selection_pipeline](#ex-10)**
- [5 - From-Scratch Implementation](#5)
    - [5.1 - ANOVA F-Statistic](#5-1)
        - **[Bonus - compute_f_statistic_from_scratch](#ex-bonus)**
- [6 - Feature Selection Report](#6)

<a name='imports'></a>
## Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
%matplotlib inline

warnings.filterwarnings('ignore', message='.*FigureCanvasAgg is non-interactive.*')

from sklearn.datasets import make_classification, make_regression, load_wine
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, SelectPercentile,
    f_classif, f_regression, chi2,
    mutual_info_classif, mutual_info_regression
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print('Libraries loaded successfully!')


<a name='1'></a>
## 1 - Filter Methods

Filter methods score each feature independently of any model. They are the fastest and most general family of feature selection techniques.

In this section you will apply three filter methods:
1. **Variance Threshold** – remove features with near-zero variance (constant features)
2. **Mutual Information** – score features by their statistical dependency with the target
3. **Visualisation** – plot variance and score distributions

<a name='1-1'></a>
### 1.1 - Load Data

We will work with three datasets:
- A **synthetic classification** dataset (many redundant features) — used for classification exercises
- A **Wine** dataset — used for correlation analysis
- A **synthetic regression** dataset — used for regression exercises

In [ ]:
# === CLASSIFICATION DATASET ===
X_clf, y_clf = make_classification(
    n_samples=600,
    n_features=30,
    n_informative=8,
    n_redundant=12,
    n_repeated=2,
    n_classes=3,
    random_state=42
)
FEATURE_NAMES_CLF = [f'feature_{i}' for i in range(X_clf.shape[1])]
print(f'Classification dataset: {X_clf.shape[0]} samples, {X_clf.shape[1]} features, {len(set(y_clf))} classes')

# === REGRESSION DATASET ===
X_reg, y_reg = make_regression(
    n_samples=400,
    n_features=20,
    n_informative=6,
    noise=0.3,
    random_state=42
)
FEATURE_NAMES_REG = [f'feature_{i}' for i in range(X_reg.shape[1])]
print(f'Regression dataset:     {X_reg.shape[0]} samples, {X_reg.shape[1]} features')

# === WINE DATASET (for correlation analysis) ===
wine = load_wine()
X_wine_df = pd.DataFrame(wine.data, columns=wine.feature_names)
y_wine = wine.target
print(f'Wine dataset:           {X_wine_df.shape[0]} samples, {X_wine_df.shape[1]} features')

In [ ]:
# Standardise datasets (needed for most statistical tests)
scaler_clf = StandardScaler()
X_clf_scaled = scaler_clf.fit_transform(X_clf)

scaler_reg = StandardScaler()
X_reg_scaled = scaler_reg.fit_transform(X_reg)

# Non-negative scaled version for chi2
mms = MinMaxScaler()
X_clf_nonneg = mms.fit_transform(X_clf)

print('Data standardised successfully.')

<a name='1-2'></a>
### 1.2 - Variance Threshold

<a name='ex-1'></a>
**Exercise 1 - apply_variance_threshold**

Implement a function that applies `VarianceThreshold` to a feature matrix.

**Instructions:**
- Create a `VarianceThreshold` selector with the given `threshold`
- Fit and transform `X`
- Return the fitted selector object and the transformed array

**Hint:** Use `sklearn.feature_selection.VarianceThreshold`

In [ ]:
def apply_variance_threshold(X, threshold=0.0):
    """
    Apply variance threshold filter to remove low-variance features.

    Args:
        X: Feature array of shape (m, n)
        threshold: Minimum variance required to keep a feature (default 0.0)

    Returns:
        selector: Fitted VarianceThreshold object
        X_filtered: Transformed array with low-variance features removed
    """
    ### START CODE HERE ### (~3 lines)
    selector = None
    X_filtered = None
    ### END CODE HERE ###

    return selector, X_filtered

In [ ]:
# Test apply_variance_threshold
# Create a dataset with one constant column and one near-zero variance column
test_X = np.column_stack([
    np.random.randn(100),      # normal feature, high variance
    np.zeros(100),             # constant feature, variance = 0
    np.random.randn(100) * 0.001,  # near-zero variance
    np.random.randn(100) * 2,  # high variance
])

sel, X_t = apply_variance_threshold(test_X, threshold=0.01)

assert sel is not None, 'selector must not be None'
assert X_t is not None, 'X_filtered must not be None'
assert X_t.shape[1] == 2, f'Expected 2 features after threshold, got {X_t.shape[1]}'

print('Exercise 1 passed!')
print(f'Original features: {test_X.shape[1]}  |  After threshold: {X_t.shape[1]}')

In [ ]:
unittests.exercise_1(apply_variance_threshold)

<a name='1-3'></a>
### 1.3 - Visualise Feature Variances

<a name='ex-2'></a>
**Exercise 2 - plot_variance_scores**

Create a horizontal bar chart of feature variances, highlighting which features fall below the threshold.

**Instructions:**
- Sort features by variance in descending order
- Show the top `top_n` features
- Draw a vertical dashed red line at the `threshold` value
- Colour bars: orange if below threshold, steelblue if above
- Add x-label `"Variance"`, y-label `"Feature"`, and a title

In [ ]:
def plot_variance_scores(variances, feature_names, threshold=0.0, top_n=20):
    """
    Plot feature variances as a horizontal bar chart.

    Args:
        variances: Array of variance values, one per feature
        feature_names: List/array of feature names
        threshold: Threshold line to draw
        top_n: Number of features to display (top n by variance)

    Returns:
        None (displays the plot)
    """
    ### START CODE HERE ### (~12 lines)

    ### END CODE HERE ###

In [ ]:
# Test plot_variance_scores — run this cell to see the plot
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.0)
vt.fit(X_clf_scaled)
feature_variances = vt.variances_

plot_variance_scores(feature_variances, FEATURE_NAMES_CLF, threshold=0.5, top_n=30)
print('Exercise 2: plot displayed above')

In [ ]:
unittests.exercise_2(plot_variance_scores)

<a name='1-4'></a>
### 1.4 - Mutual Information Selection

<a name='ex-3'></a>
**Exercise 3 - select_with_mutual_information**

Select the top `k` features using mutual information.

**Instructions:**
- Use `SelectKBest` with `mutual_info_classif` for classification task types, or `mutual_info_regression` for regression
- Fit on `(X, y)` and transform `X`
- Return the selector and the selected feature array

**Hint:** `task_type` will be either `'classification'` or `'regression'`

In [ ]:
def select_with_mutual_information(X, y, k, task_type='classification'):
    """
    Select k best features using mutual information.

    Args:
        X: Feature array
        y: Target array
        k: Number of features to select
        task_type: 'classification' or 'regression'

    Returns:
        selector: Fitted SelectKBest object
        X_selected: Transformed array with k features
    """
    ### START CODE HERE ### (~6 lines)
    selector = None
    X_selected = None
    ### END CODE HERE ###

    return selector, X_selected

In [ ]:
# Test select_with_mutual_information
k = 8
sel_mi, X_mi = select_with_mutual_information(X_clf_scaled, y_clf, k=k)

assert sel_mi is not None, 'selector must not be None'
assert X_mi is not None, 'X_selected must not be None'
assert X_mi.shape == (X_clf_scaled.shape[0], k), \
    f'Expected shape ({X_clf_scaled.shape[0]}, {k}), got {X_mi.shape}'

selected_names = [FEATURE_NAMES_CLF[i] for i in range(len(FEATURE_NAMES_CLF)) if sel_mi.get_support()[i]]
print('Exercise 3 passed!')
print(f'Selected {k} features: {selected_names}')

In [ ]:
unittests.exercise_3(select_with_mutual_information)

<a name='2'></a>
## 2 - Statistical Tests Selection

Statistical tests give you a **score and a p-value** for each feature, measuring how strongly it relates to the target.

Key rule: **choose the test that matches your problem type:**
- Classification + continuous features → `f_classif`
- Classification + non-negative features → `chi2`
- Regression + continuous features → `f_regression`

<a name='2-1'></a>
### 2.1 - F-test for Classification

<a name='ex-4'></a>
**Exercise 4 - select_with_f_classif**

Select the top `k` features for a classification problem using the ANOVA F-test.

**Instructions:**
- Use `SelectKBest` with `f_classif`
- Build and return a **scores DataFrame** with columns: `'feature'`, `'F_score'`, `'p_value'`, sorted by `'F_score'` descending
- Return the fitted selector, the selected feature array, and the scores DataFrame

In [ ]:
def select_with_f_classif(X, y, k, feature_names):
    """
    Select k best features for classification using ANOVA F-test.

    Args:
        X: Feature array (continuous values)
        y: Classification target
        k: Number of features to select
        feature_names: List of feature names

    Returns:
        selector: Fitted SelectKBest object
        X_selected: Transformed array with k features
        scores_df: DataFrame with columns ['feature','F_score','p_value'],
                   sorted by F_score descending
    """
    ### START CODE HERE ### (~8 lines)
    selector = None
    X_selected = None
    scores_df = None
    ### END CODE HERE ###

    return selector, X_selected, scores_df

In [ ]:
# Test select_with_f_classif
k = 10
sel_f, X_f, scores_f = select_with_f_classif(X_clf_scaled, y_clf, k, FEATURE_NAMES_CLF)

assert sel_f is not None, 'selector must not be None'
assert X_f.shape == (X_clf_scaled.shape[0], k), f'Expected shape ({X_clf_scaled.shape[0]}, {k})'
assert scores_f is not None, 'scores_df must not be None'
assert set(scores_f.columns) >= {'feature', 'F_score', 'p_value'}, 'Missing columns in scores_df'
assert list(scores_f['F_score']) == sorted(scores_f['F_score'], reverse=True), 'scores_df not sorted'

print('Exercise 4 passed!')
print(f'\nTop 10 features by F-score:')
print(scores_f.head(10).to_string(index=False))

In [ ]:
unittests.exercise_4(select_with_f_classif)

<a name='2-2'></a>
### 2.2 - Chi-Square Test

<a name='ex-5'></a>
**Exercise 5 - select_with_chi2**

Select the top `k` features using the chi-square test.

**Important:** `chi2` requires all feature values to be **non-negative**. Use `X_clf_nonneg` (already scaled with `MinMaxScaler`).

**Instructions:**
- Use `SelectKBest` with `chi2`
- Input `X_nonneg` must already have non-negative values
- Return the fitted selector and the selected feature array

In [ ]:
def select_with_chi2(X_nonneg, y, k):
    """
    Select k best features using the chi-square test.

    Args:
        X_nonneg: Feature array with non-negative values (e.g., after MinMaxScaler)
        y: Classification target
        k: Number of features to select

    Returns:
        selector: Fitted SelectKBest object
        X_selected: Transformed array with k features
    """
    ### START CODE HERE ### (~3 lines)
    selector = None
    X_selected = None
    ### END CODE HERE ###

    return selector, X_selected

In [ ]:
# Test select_with_chi2
k = 10
sel_chi2, X_chi2 = select_with_chi2(X_clf_nonneg, y_clf, k)

assert sel_chi2 is not None, 'selector must not be None'
assert X_chi2 is not None, 'X_selected must not be None'
assert X_chi2.shape == (X_clf_nonneg.shape[0], k), \
    f'Expected shape ({X_clf_nonneg.shape[0]}, {k}), got {X_chi2.shape}'
assert np.all(X_chi2 >= 0), 'All values should be non-negative (chi2 requires this)'

print('Exercise 5 passed!')
print(f'chi2 selected {k} features out of {X_clf_nonneg.shape[1]}')

In [ ]:
unittests.exercise_5(select_with_chi2)

<a name='2-3'></a>
### 2.3 - F-test for Regression

<a name='ex-6'></a>
**Exercise 6 - select_for_regression**

Select the top `k` features for a **regression** problem using the F-test.

**Instructions:**
- Use `SelectKBest` with `f_regression`
- Return the fitted selector, selected feature array, and a scores DataFrame with columns `['feature', 'F_score', 'p_value']` sorted by F_score descending

In [ ]:
def select_for_regression(X, y, k, feature_names):
    """
    Select k best features for regression using the F-test.

    Args:
        X: Feature array (continuous values)
        y: Continuous regression target
        k: Number of features to select
        feature_names: List of feature names

    Returns:
        selector: Fitted SelectKBest object
        X_selected: Transformed array with k features
        scores_df: DataFrame with columns ['feature','F_score','p_value'],
                   sorted by F_score descending
    """
    ### START CODE HERE ### (~8 lines)
    selector = None
    X_selected = None
    scores_df = None
    ### END CODE HERE ###

    return selector, X_selected, scores_df

In [ ]:
# Test select_for_regression
k = 6
sel_reg, X_reg_s, scores_reg = select_for_regression(X_reg_scaled, y_reg, k, FEATURE_NAMES_REG)

assert sel_reg is not None, 'selector must not be None'
assert X_reg_s.shape == (X_reg_scaled.shape[0], k)
assert scores_reg is not None
assert list(scores_reg['F_score']) == sorted(scores_reg['F_score'], reverse=True)

print('Exercise 6 passed!')
print(f'\nTop features for regression:')
print(scores_reg.head(k).to_string(index=False))

In [ ]:
unittests.exercise_6(select_for_regression)

<a name='2-4'></a>
### 2.4 - Compare Selection Methods

<a name='ex-7'></a>
**Exercise 7 - compare_selection_methods**

Compare feature rankings from `f_classif` and `mutual_info_classif`. The two methods sometimes disagree — understanding when and why is important.

**Instructions:**
- Compute scores using both `f_classif` and `mutual_info_classif` with `k='all'`
- Assign ranks (1 = best) to each feature from each method
- Return a DataFrame with columns: `'feature'`, `'f_classif_score'`, `'mi_score'`, `'f_rank'`, `'mi_rank'`, `'rank_diff'` (absolute difference between ranks), sorted by `'f_rank'`

In [ ]:
def compare_selection_methods(X, y, feature_names):
    """
    Compare feature rankings from f_classif and mutual_info_classif.

    Args:
        X: Feature array
        y: Classification target
        feature_names: List of feature names

    Returns:
        comparison_df: DataFrame with columns:
            ['feature', 'f_classif_score', 'mi_score',
             'f_classif_rank', 'mi_rank', 'rank_diff']
            sorted by f_classif_rank ascending
    """
    ### START CODE HERE ### (~15 lines)
    comparison_df = None
    ### END CODE HERE ###

    return comparison_df

In [ ]:
# Test compare_selection_methods
comparison = compare_selection_methods(X_clf_scaled, y_clf, FEATURE_NAMES_CLF)

assert comparison is not None, 'comparison_df must not be None'
expected_cols = {'feature', 'f_classif_score', 'mi_score', 'f_rank', 'mi_rank', 'rank_diff'}\n",
assert expected_cols.issubset(set(comparison.columns)), f'Missing columns: {expected_cols - set(comparison.columns)}'
assert len(comparison) == len(FEATURE_NAMES_CLF), 'DataFrame should have one row per feature'

print('Exercise 7 passed!')
print('\nFeatures where methods disagree most (largest rank difference):')
print(comparison.nlargest(5, 'rank_diff')[['feature','f_rank','mi_rank','rank_diff']].to_string(index=False))

In [ ]:
unittests.exercise_7(compare_selection_methods)

<a name='3'></a>
## 3 - Correlation-Based Elimination

Statistical tests tell you which features relate to the **target**. But they do not tell you when two features relate to **each other** (redundancy).

Correlation-based elimination finds highly correlated feature pairs and removes the less relevant one from each pair.

In [ ]:
# Visualise the wine feature correlation matrix
corr_matrix = X_wine_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Wine Dataset — Feature Correlation Matrix (lower triangle)')
plt.tight_layout()
plt.show()
print('High values (close to ±1) indicate redundant feature pairs.')

<a name='3-1'></a>
### 3.1 - Find Correlated Pairs

<a name='ex-8'></a>
**Exercise 8 - find_correlated_pairs**

Identify all pairs of features whose absolute Pearson correlation exceeds a threshold.

**Instructions:**
- Compute the absolute correlation matrix from `X_df`
- Look only at the **upper triangle** to avoid duplicate pairs
- Collect tuples `(feature_a, feature_b, correlation_value)` for all pairs above `threshold`
- Return the list sorted by correlation descending

In [ ]:
def find_correlated_pairs(X_df, threshold=0.85):
    """
    Find all feature pairs with absolute correlation above the threshold.

    Args:
        X_df: pandas DataFrame of features
        threshold: Minimum absolute correlation to flag a pair

    Returns:
        correlated_pairs: List of tuples (feature_a, feature_b, corr_value)
                          sorted by corr_value descending
    """
    ### START CODE HERE ### (~10 lines)
    correlated_pairs = []
    ### END CODE HERE ###

    return correlated_pairs

In [ ]:
# Test find_correlated_pairs on wine dataset
pairs = find_correlated_pairs(X_wine_df, threshold=0.75)

assert isinstance(pairs, list), 'Return value must be a list'
assert len(pairs) > 0, 'Expected at least one correlated pair in wine dataset above 0.75'
for p in pairs:
    assert len(p) == 3, 'Each element must be a 3-tuple (feat_a, feat_b, corr)'
    assert abs(p[2]) >= 0.75, f'Correlation {p[2]:.3f} below threshold'

# Check sorted descending
corr_values = [abs(p[2]) for p in pairs]
assert corr_values == sorted(corr_values, reverse=True), 'Pairs must be sorted by correlation descending'

print('Exercise 8 passed!')
print(f'\nFound {len(pairs)} correlated pairs (|r| > 0.75):')
for feat_a, feat_b, corr in pairs[:5]:
    print(f'  {feat_a[:20]:20s}  <->  {feat_b[:20]:20s}  r = {corr:.3f}')

In [ ]:
unittests.exercise_8(find_correlated_pairs)

<a name='3-2'></a>
### 3.2 - Remove Correlated Features

<a name='ex-9'></a>
**Exercise 9 - remove_correlated_features**

Greedy algorithm: for each correlated pair, remove the feature with the **lower relevance score** to the target.

**Instructions:**
- Compute absolute correlation matrix and look at the upper triangle only
- For each pair with correlation > threshold, add the feature with the **lower** `target_scores` value to the removal set
- Return two lists: features kept and features removed

In [ ]:
def remove_correlated_features(X_df, target_scores, threshold=0.85):
    """
    Greedy removal of correlated features.
    Keeps the feature with the higher target relevance score from each pair.

    Args:
        X_df: pandas DataFrame of features
        target_scores: dict {feature_name: relevance_score}
        threshold: Absolute correlation threshold

    Returns:
        features_to_keep: List of feature names to retain
        features_removed: List of feature names that were removed
    """
    ### START CODE HERE ### (~15 lines)
    features_to_keep = []
    features_removed = []
    ### END CODE HERE ###

    return features_to_keep, features_removed

In [ ]:
# Test remove_correlated_features on wine dataset
# First get F-scores as relevance metric
f_sel = SelectKBest(f_classif, k='all')
f_sel.fit(X_wine_df.values, y_wine)
wine_scores = dict(zip(X_wine_df.columns, f_sel.scores_))

threshold = 0.75
kept, removed = remove_correlated_features(X_wine_df, wine_scores, threshold=threshold)

assert isinstance(kept, list)
assert isinstance(removed, list)
assert len(kept) + len(removed) == X_wine_df.shape[1], 'kept + removed must equal total features'
assert len(set(kept) & set(removed)) == 0, 'A feature cannot be both kept and removed'

# Verify no remaining pair exceeds threshold
X_kept = X_wine_df[kept]
corr_kept = X_kept.corr().abs()
upper = corr_kept.where(np.triu(np.ones(corr_kept.shape), k=1).astype(bool))
max_corr = upper.stack().max() if not upper.stack().empty else 0
assert max_corr <= threshold or np.isnan(max_corr), \
    f'Found remaining pair with correlation {max_corr:.3f} > {threshold}'

print('Exercise 9 passed!')
print(f'\nWine dataset ({X_wine_df.shape[1]} features) → kept {len(kept)}, removed {len(removed)}')
print(f'Removed: {removed}')

In [ ]:
unittests.exercise_9(remove_correlated_features)

<a name='4'></a>
## 4 - Feature Selection Pipeline

Using a `Pipeline` ensures that the selector is fitted **only on training data**, preventing data leakage.

The pipeline will chain: `StandardScaler → SelectKBest → LogisticRegression`

<a name='4-1'></a>
### 4.1 - Build Complete Pipeline

<a name='ex-10'></a>
**Exercise 10 - create_selection_pipeline**

Build and return an sklearn `Pipeline` with three steps:
1. `('scaler', StandardScaler())`
2. `('selector', SelectKBest(score_func, k=k))`
3. `('model', LogisticRegression(max_iter=1000))`

**Instructions:**
- Return an **unfitted** Pipeline object
- Use the `k` and `score_func` arguments as parameters

In [ ]:
def create_selection_pipeline(k, score_func=f_classif):
    """
    Build a feature selection pipeline:
    StandardScaler -> SelectKBest -> LogisticRegression

    Args:
        k: Number of features to select
        score_func: Scoring function for SelectKBest (default: f_classif)

    Returns:
        pipeline: Unfitted sklearn Pipeline
    """
    ### START CODE HERE ### (~6 lines)
    pipeline = None
    ### END CODE HERE ###

    return pipeline

In [ ]:
# Test create_selection_pipeline
pipe = create_selection_pipeline(k=10, score_func=f_classif)

assert pipe is not None, 'Pipeline must not be None'
assert hasattr(pipe, 'fit'), 'Return value must be a Pipeline'
assert 'scaler' in pipe.named_steps, 'Pipeline must contain a scaler step'
assert 'selector' in pipe.named_steps, 'Pipeline must contain a selector step'
assert 'model' in pipe.named_steps, 'Pipeline must contain a model step'

# Fit and evaluate to verify it works end-to-end
X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)
pipe.fit(X_tr, y_tr)
accuracy = pipe.score(X_te, y_te)

print('Exercise 10 passed!')
print(f'Pipeline test accuracy: {accuracy:.3f}')

# Compare k values using cross-validation
k_values = [5, 8, 10, 15, 20, 25, 30]
cv_scores = []
for k_val in k_values:
    p = create_selection_pipeline(k=k_val)
    score = cross_val_score(p, X_clf, y_clf, cv=5, scoring='accuracy').mean()
    cv_scores.append(score)

plt.figure(figsize=(8, 4))
plt.plot(k_values, cv_scores, marker='o', color='steelblue')
plt.xlabel('Number of selected features (k)')
plt.ylabel('5-fold CV Accuracy')
plt.title('Feature Count vs Model Performance')
plt.xticks(k_values)
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_10(create_selection_pipeline)

<a name='5'></a>
## 5 - From-Scratch Implementation

Implementing the ANOVA F-statistic from scratch builds deeper intuition for what `f_classif` does internally.

**Recall the formula:**
$$F_j = \\frac{\\text{between-group variance of feature } j}{\\text{within-group variance of feature } j}$$

where:
- **Between-group variance** = $\\sum_c n_c (\\bar{x}_{c,j} - \\bar{x}_j)^2 \\;/\; (K-1)$
- **Within-group variance** = $\\sum_c \\sum_{i \\in c} (x_{i,j} - \\bar{x}_{c,j})^2 \\;/\; (N-K)$

($K$ = number of classes, $N$ = number of samples, $\\bar{x}_{c,j}$ = mean of feature $j$ in class $c$)

<a name='5-1'></a>
### 5.1 - ANOVA F-Statistic from Scratch

<a name='ex-bonus'></a>
**Bonus Exercise - compute_f_statistic_from_scratch**

Implement the ANOVA F-statistic calculation for each feature independently.

**Instructions:**
- Compute the overall mean of each feature across all samples (`overall_mean`)
- For each class `c`, compute `class_mean_c` (mean of each feature within class `c`) and `n_c` (number of samples)
- Compute between-group variance and within-group variance using the formulas above
- Return an array of F-scores, one per feature

In [ ]:
def compute_f_statistic_from_scratch(X, y):
    """
    Compute the ANOVA F-statistic for each feature manually.

    Args:
        X: Feature matrix of shape (n_samples, n_features), continuous values
        y: Integer class labels of shape (n_samples,)

    Returns:
        f_scores: numpy array of F-scores, shape (n_features,)
    """
    n_samples, n_features = X.shape
    classes = np.unique(y)
    K = len(classes)

    ### START CODE HERE ### (~15 lines)
    f_scores = None
    ### END CODE HERE ###

    return f_scores

In [ ]:
# Test compute_f_statistic_from_scratch
f_manual = compute_f_statistic_from_scratch(X_clf_scaled, y_clf)

if f_manual is not None:
    # Compare with sklearn's f_classif
    f_sklearn, _ = f_classif(X_clf_scaled, y_clf)

    assert f_manual.shape == (X_clf_scaled.shape[1],), \
        f'Expected shape ({X_clf_scaled.shape[1]},), got {f_manual.shape}'
    assert np.allclose(f_manual, f_sklearn, rtol=1e-3), \
        'F-scores do not match sklearn. Check your formula.'

    print('Bonus exercise passed!')
    print(f'Max absolute error vs sklearn: {np.max(np.abs(f_manual - f_sklearn)):.6f}')
else:
    print('Bonus not yet implemented — skipping comparison')

In [ ]:
unittests.exercise_11(compute_f_statistic_from_scratch)

<a name='6'></a>
## 6 - Feature Selection Report

Document your feature selection process by filling in the template below.

A good report answers:
1. How many features did you **start with**?
2. Which features were **removed** at each step and why?
3. What was the **effect on model performance**?

---

### Your Report

*(Edit this cell to fill in your findings)*

**Dataset:** Synthetic classification (30 features, 600 samples)

**Step 1 – Variance Threshold**
- Threshold used: `___`
- Features removed: `___`
- Reason: ...

**Step 2 – Statistical Test (f_classif)**
- k selected: `___`
- Top 3 features (by F-score): `___`
- Features with p > 0.05: `___`

**Step 3 – Correlation Elimination**
- Threshold used: `___`
- Pairs found: `___`
- Features removed: `___`

**Model Comparison**

| Setup | 5-fold CV Accuracy |
|---|---|
| All 30 features | ??? |
| After feature selection | ??? |

**Conclusion:** ...